## 1. Basic Tasks

**1. Use MERGE INTO to upsert a batch of 'changed customer' records into a silver customers table.**

In [0]:
df = spark.read.csv(
    "/Volumes/dev/bronze/raw/customer_new.csv",
    header=True,
    inferSchema=True
)
df.write.mode("overwrite").saveAsTable("dev.bronze.customers_table")

In [0]:
%sql
MERGE INTO dev.silver.customers_clean as t
USING dev.bronze.customers_table as s
ON t.customer_id = s.customer_id
WHEN MATCHED THEN 
    UPDATE SET
        t.name = s.name,
        t.email = s.email,
        t.city = s.city,
        t.state = s.state,
        t.signup_date = s.signup_date,
        t.phone = s.phone
WHEN NOT MATCHED THEN
    INSERT (customer_id, name, email, city, state, signup_date, phone)
    VALUES (s.customer_id, s.name, s.email, s.city, s.state, s.signup_date, s.phone)


**2. Grant SELECT on a table to one group and a masked/limited view to another group using Unity
Catalog permissions.**

In [0]:
%sql

CREATE OR REPLACE VIEW dev.silver.customers_restricted AS
SELECT 
    customer_id,
    name,
    CONCAT(LEFT(email, LENGTH(email) - 10),REPEAT("*", 10)) AS email,
    city,
    state,
    signup_date,
    CONCAT(LEFT(phone, LENGTH(phone) - 3),"***") AS phone
FROM dev.silver.customers_clean;

In [0]:
%sql
GRANT USE CATALOG
ON CATALOG dev
TO `data_analysts`;

GRANT USE SCHEMA
ON SCHEMA dev.silver
TO `data_analysts`;

GRANT USE CATALOG
ON CATALOG dev
TO `data_engineers`;

GRANT USE SCHEMA
ON SCHEMA dev.silver
TO `data_engineers`;

In [0]:
%sql
GRANT SELECT 
ON VIEW dev.silver.customers_restricted
TO `data_analysts`;

GRANT SELECT 
ON VIEW dev.silver.customers_cleaned
TO `data_engineers`;

**3. Look up the DBU consumption for a compute resource you've been using and explain, in plain terms,
what a DBU is billing for.**

A Databricks Unit (DBU) is a normalized unit of processing power on the Databricks Lakehouse Platform used for measurement and pricing purposes. The number of DBUs a workload consumes is driven by processing metrics, which may include the compute resources used and the amount of data processed.

To check DBU consumption you need to go to Accounts and then Uses, there you can find the DBU uses and estimated cost across your Workspaces.

## 2. Intermediate Tasks

**4. Build a full SCD Type 2 table: implement the MERGE that closes out old records (setting end_date
and is_current) and inserts new versions when a tracked column changes.**

In [0]:
%sql
-- Target Table
CREATE TABLE dev.silver.dim_customer (
    customer_id INT,
    name VARCHAR(255),
    email VARCHAR(255),
    city VARCHAR(100),
    state VARCHAR(50),
    signup_date DATE,
    phone LONG,
    start_date DATE,
    end_date DATE,
    is_current BOOLEAN
);

In [0]:
# Source Table
df = spark.read.csv(
    "/Volumes/dev/bronze/raw/customers/customers_csv/customers.csv",
    header=True,
    inferSchema=True
)
df1 = df.dropDuplicates().dropna(how="any")
df1.write.mode("overwrite").saveAsTable("dev.bronze.stg_customer")

In [0]:
%sql
MERGE INTO dev.silver.dim_customer AS tgt
USING (
    SELECT customer_id, name, email, city, state, signup_date, phone,
        'merge_update' AS merge_action
    FROM dev.bronze.stg_customer
    
    UNION ALL
    
    SELECT src.customer_id, src.name, src.email, src.city, src.state, src.signup_date, src.phone,
        'merge_insert' AS merge_action
    FROM dev.bronze.stg_customer src
    JOIN dev.silver.dim_customer tgt ON src.customer_id = tgt.customer_id AND tgt.is_current = TRUE
    WHERE 
        COALESCE(src.name, '') <> COALESCE(tgt.name, '') OR
        COALESCE(src.email, '') <> COALESCE(tgt.email, '') OR
        COALESCE(src.city, '') <> COALESCE(tgt.city, '') OR
        COALESCE(src.state, '') <> COALESCE(tgt.state, '') OR
        COALESCE(src.phone, '') <> COALESCE(tgt.phone, '')
) AS src

ON tgt.customer_id = src.customer_id 
   AND tgt.is_current = TRUE 
   AND src.merge_action = 'merge_update'

WHEN MATCHED AND (
    COALESCE(tgt.name, '') <> COALESCE(src.name, '') OR
    COALESCE(tgt.email, '') <> COALESCE(src.email, '') OR
    COALESCE(tgt.city, '') <> COALESCE(src.city, '') OR
    COALESCE(tgt.state, '') <> COALESCE(src.state, '') OR
    COALESCE(tgt.phone, '') <> COALESCE(src.phone, '')
) THEN
    UPDATE SET 
        tgt.end_date = CURRENT_DATE,
        tgt.is_current = FALSE

WHEN NOT MATCHED THEN
    INSERT (customer_id, name, email, city, state, signup_date, phone, start_date, end_date, 
        is_current
    )
    VALUES (
        src.customer_id, src.name, src.email, src.city, src.state, src.signup_date, src.phone, 
        CURRENT_DATE, '9999-12-31', TRUE
    );

**5. Query the SCD Type 2 table to answer a point-in-time question, e.g. 'what was this customer's
address as of March 1st?'**

In [0]:
%sql
SELECT 
CONCAT(city, ', ', state) AS address
FROM dev.silver.dim_customer
WHERE signup_date = '2023-08-02'

**6. Compare the estimated DBU cost of running a job on all-purpose vs. job compute, and recommend
which Cyntexa should use for its nightly pipeline.**

All-Purpose Compute ($0.40–$0.55/DBU) is **2.5x to 4x more expensive** than Job Compute (~$0.15/DBU).


For Cyntexa we should strictly use **Job Compute** for its nightly pipeline.

**Because:**

* **Massive Savings:** We can cut the active compute costs by 60–75%.
* **Zero Idle Spend:** Job cluster gets terminate the exact second the pipeline finishes. So that's why we should use it, cause All-Purpose clusters sit idle and burn money until their auto-terminate timer hits.
* **Reliability:** Job clusters provide a fresh, conflict-free environment for every run.

**Action:** When scheduling the Databricks Workflow, configure the task to use a **"New Job Cluster"** instead of attaching an existing All-Purpose cluster.

## 3. Advanced Tasks

**7. Design a governance model for Cyntexa: which columns across which tables are sensitive (PII), which
Unity Catalog groups should have access, and how you'd audit access after the fact.**